In [62]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, r2_score, mean_absolute_error

In [63]:
2# Lab 10: Data Preprocessing Steps

# 1. Reading csv file
data = pd.read_csv('diabetes.csv')
print("CSV file loaded successfully!")

CSV file loaded successfully!


In [64]:
# 2. Printing rows and columns / shape of dataframe
print('We have {} rows.'.format(data.shape[0]))
print('We have {} columns'.format(data.shape[1]))
print("Data shape:", data.shape)

We have 768 rows.
We have 9 columns
Data shape: (768, 9)


In [65]:
# 3. Checking null values
print("Null values in each column:")
print(np.sum(pd.isnull(data)))

Null values in each column:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


c:\Users\HS TRADER\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [66]:
# 4. Checking unique values of specific column
print("Unique values in 'Outcome' column:")
print(data['Outcome'].unique())

Unique values in 'Outcome' column:
[1 0]


In [67]:
# 5. Filling null values
# Check if there are null values in 'Outcome' column
if data['Outcome'].isnull().sum() > 0:
    mode_value = data['Outcome'].mode()[0]
    data['Outcome'] = data['Outcome'].fillna(mode_value)
    print(f"Filled null values in 'Outcome' with mode: {mode_value}")
else:
    print("No null values found in 'Outcome' column")

No null values found in 'Outcome' column


In [68]:
# 6. Type conversion to int
# Convert float columns to int where appropriate
float_columns = data.select_dtypes(include=['float64']).columns
for col in float_columns:
    data[col] = data[col].astype(np.int64)
print("Converted float columns to int:")
print(data.dtypes)

Converted float columns to int:
Pregnancies                 int64
Glucose                     int64
BloodPressure               int64
SkinThickness               int64
Insulin                     int64
BMI                         int64
DiabetesPedigreeFunction    int64
Age                         int64
Outcome                     int64
dtype: object


In [69]:
# 7. Dropping unnecessary columns
# Check columns and drop if there's an 'id' or similar unnecessary column
if 'id' in data.columns:
    data.drop('id', axis=1, inplace=True)
    print("Dropped 'id' column")
else:
    print("No 'id' column found to drop")

No 'id' column found to drop


In [70]:
# 8. Check datatypes of all columns
print("Data types of all columns:")
print(data.dtypes)

Data types of all columns:
Pregnancies                 int64
Glucose                     int64
BloodPressure               int64
SkinThickness               int64
Insulin                     int64
BMI                         int64
DiabetesPedigreeFunction    int64
Age                         int64
Outcome                     int64
dtype: object


In [71]:
# 9. Splitting into x and y
x = data.iloc[:, 0:-1]
print("Features (x) shape:", x.shape)
y = data.iloc[:, -1]
print("Target (y) shape:", y.shape)

Features (x) shape: (768, 8)
Target (y) shape: (768,)


In [72]:
# 10. Converting Object columns into Int columns
cat_columns = x.select_dtypes(['object']).columns
if len(cat_columns) > 0:
    x[cat_columns] = x[cat_columns].apply(lambda col: pd.factorize(col)[0])
    print("Converted object columns to int:", list(cat_columns))
else:
    print("No object columns found to convert")

print("Final data types after preprocessing:")
print(x.dtypes)
print(y.dtypes)

No object columns found to convert
Final data types after preprocessing:
Pregnancies                 int64
Glucose                     int64
BloodPressure               int64
SkinThickness               int64
Insulin                     int64
BMI                         int64
DiabetesPedigreeFunction    int64
Age                         int64
dtype: object
int64


In [73]:
# Use the preprocessed data from previous cells
features_train, features_test, target_train, target_test = train_test_split(x, y, test_size=0.2, random_state=42)
normalizer = StandardScaler()
features_train_normalized = normalizer.fit_transform(features_train)
features_test_normalized = normalizer.transform(features_test)

In [74]:
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(features_train_normalized, target_train)
with open('model/diabetes_classifier.pkl', 'wb') as file:
    pickle.dump(classifier, file)
with open('model/feature_normalizer.pkl', 'wb') as file:
    pickle.dump(normalizer, file)

In [75]:
target_pred_test = classifier.predict(features_test_normalized)
sample_1 = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])
sample_1_normalized = normalizer.transform(sample_1)
risk_1 = classifier.predict(sample_1_normalized)
sample_2 = np.array([[1, 85, 66, 29, 0, 26.6, 0.351, 31]])
sample_2_normalized = normalizer.transform(sample_2)
risk_2 = classifier.predict(sample_2_normalized)
print("=== Personalized Diabetes Risk Assessments ===")
print(f"Assessment 1: Pregnancies=6, Glucose=148, BP=72, Skin=35, Insulin=0, BMI=33.6, Pedigree=0.627, Age=50")
print(f"Diabetes Risk: {'High' if risk_1[0] == 1 else 'Low'}")
print(f"Assessment 2: Pregnancies=1, Glucose=85, BP=66, Skin=29, Insulin=0, BMI=26.6, Pedigree=0.351, Age=31")
print(f"Diabetes Risk: {'High' if risk_2[0] == 1 else 'Low'}")
print("=== System Performance ===")
accuracy = accuracy_score(target_test, target_pred_test)
print(f"Precision Score: {accuracy:.4f}")
print(f"Success Rate: {accuracy * 100:.2f}%")

=== Personalized Diabetes Risk Assessments ===
Assessment 1: Pregnancies=6, Glucose=148, BP=72, Skin=35, Insulin=0, BMI=33.6, Pedigree=0.627, Age=50
Diabetes Risk: High
Assessment 2: Pregnancies=1, Glucose=85, BP=66, Skin=29, Insulin=0, BMI=26.6, Pedigree=0.351, Age=31
Diabetes Risk: Low
=== System Performance ===
Precision Score: 0.7468
Success Rate: 74.68%


c:\Users\HS TRADER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\HS TRADER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
